In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

from ase.io import read, write
from ase.neighborlist import NeighborList

from itertools import combinations

In [3]:
def get_phi(angle1, angle2, angle1_std, angle2_std):
    ''' 
     order parameters for perovskites are from Keefe et al. Acta Cryst., 1977. B33, 3802-3813.
     phi1 uses the larger angle of the two crystallographically unequivalent BXB angles
     phi2 uses the smaller angle
     returns the mean of the two order parameters 
     example value get_phi(155.63, 155.12) gives 15.056 (replicating value from Thygesen et al. Phys. Rev. B, 2017 95, 174107.)
    '''
    
    phi1 = np.acos(np.sqrt((2-2*np.cos(angle1*np.pi/180))/(5+np.cos(angle1*np.pi/180))))*180/np.pi
    phi1_std = angle1_std*2*phi1/angle1
    phi2 = np.acos(np.sqrt((1-3*np.cos(angle2*np.pi/180))/4))*180/np.pi
    phi2_std = angle2_std*phi2/angle2_std
    return [(phi1+phi2)/2, (phi1_std+phi2_std)]

In [4]:
def find_XX_vectors(atom, B_indices, X_indices, cutoff):

    '''
    Finds the vectors between two atoms X that are bonded to the same atom B, so basically the vector between the opposing X atoms in an octahedra in our case.
    Returns a list of vectors between the two X atoms, the index of the two X atoms, and the index of the B atom.
    '''
    
    cutoffs = [cutoff/2] * len(atom)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True, skin=0.0)
    nl.update(atom)
    
    XX_vectors = []
    for B_index in B_indices:
        index, offsets = nl.get_neighbors(B_index)
        BX_bond = []
        for X_index, offset in zip(index, offsets):
            if X_index in X_indices:
                vec = atom.positions[X_index] + np.dot(offset, atom.get_cell()) - atom.positions[B_index]
                BX_bond.append([vec, X_index])
                
        for pair in combinations(BX_bond, 2):
            if -1 > np.dot(pair[0][0], pair[1][0]):
                XX_vectors.append([pair[0][0]-pair[1][0], pair[0][1], pair[1][1]]) # rugged way of finding parallel vectors

    return XX_vectors

In [5]:
def find_XXX_angles(X_indices, XX_vectors):

    '''
    Finds the angles between two XX_vectors that are connected to the same atom X.
    Returns a list of angles in the x, y, and z direction.
    '''


    x_angles = []
    y_angles = []
    z_angles = []
    
    for X_index in X_indices:
        XXX_vectors = [i for i in XX_vectors if i[1] == X_index or i[2] == X_index] # find connected XX vectors
        if len(XXX_vectors) == 2:
            angle = np.degrees(np.arccos(np.dot(XXX_vectors[0][0], XXX_vectors[1][0]) / (np.linalg.norm(XXX_vectors[0][0]) * np.linalg.norm(XXX_vectors[1][0])))) # find XXX angle
            
            if angle < 90:
                angle = 180 - angle # make sure angle is always obtuse

            # check if angle is in x, y, or z direction
                
            if -2 > np.dot(XXX_vectors[0][0], [1,0,0]) or np.dot(XXX_vectors[0][0], [1,0,0]) > 2: 
                x_angles.append(angle)
                
            if -2 > np.dot(XXX_vectors[0][0], [0,1,0]) or np.dot(XXX_vectors[0][0], [0,1,0]) > 2:
                y_angles.append(angle)
                
            if -2 > np.dot(XXX_vectors[0][0], [0,0,1]) or np.dot(XXX_vectors[0][0], [0,0,1]) > 2:
                z_angles.append(angle)

    return x_angles, y_angles, z_angles

In [6]:
atoms = read('coded_JT_smooth_10.xyz', index=':')

In [7]:
Bsite = 'Mn'
Xsite = 'O'

B_indices = [site.index for site in atoms[0] if site.symbol == Bsite]
X_indices = [site.index for site in atoms[0] if site.symbol == Xsite]

cutoff = 3.0

x = []
y = []
z = []

for atom in atoms:
    x_angles, y_angles, z_angles = find_XXX_angles(X_indices, find_XX_vectors(atom, B_indices, X_indices, cutoff))
    x.append(x_angles)
    y.append(y_angles)
    z.append(z_angles)


# Saving just in case, because this takes so long for all the structures
np.savez('XXX_angles_x_smooth_10.npz', *x)
np.savez('XXX_angles_y_smooth_10.npz', *y)
np.savez('XXX_angles_z_smooth_10.npz', *z)

In [5]:
# Loading the list of arrays from the .npz file
loaded = np.load('XXX_angles_x.npz')
x = [loaded[f'arr_{i}'] for i in range(len(loaded.files))]

loaded = np.load('XXX_angles_y.npz')
y = [loaded[f'arr_{i}'] for i in range(len(loaded.files))]

loaded = np.load('XXX_angles_z.npz')
z = [loaded[f'arr_{i}'] for i in range(len(loaded.files))]

In [15]:
# averaging across all octahedra in each structure

x_mean = []
x_std = []
y_mean = []
y_std = []
z_mean = []
z_std = []

for i, j, k in zip(x, y, z):
    x_mean.append(np.mean(i, axis=0))
    y_mean.append(np.mean(j, axis=0))
    z_mean.append(np.mean(k, axis=0))
    x_std.append(np.std(i, axis=0))
    y_std.append(np.std(j, axis=0))
    z_std.append(np.std(k, axis=0))

In [10]:
xray_x = [323.71920093457913,372.67554827586184,583.3017311848746, 595.2561815286281, 605.5028780696098, 612.3339801428724, 624.8576817542164, 631.6888272585668, 644.2125288699108, 652.182263771618,  661.2903274841551, 672.6755265603178, 683.4915177999779, 693.1689196422813, 703.9848240197657, 711.9545154903851, 723.339714566548,  733.586324245354,  742.6945182511546, 752.3719200934579, 760.9108194005801, 774.5731104092808, 783.6812175529058, 793.9279140938875, 804.7438184713719, 813.2827177784941, 824.6679168546568, 836.0531159308198, 845.1613099366202, 854.8387117789237, 863.3776110860457, 873.6243076270274, 885.5788014018685, 890.1328115425933, 902.0873053174346, 915.7495094639597, 925.4269981684386, 932.827308078204,  944.7818018530452, 954.4592036953485, 964.1366055376518, 973.8140942421307, 982.9222013857557, 993.1688979267374]

xray_y = [15.02990658377151,14.94018690375972,14.65233634690841,14.60000005704220,14.57383180515496,14.54018688949917,14.52523370456753,14.50280371326183,14.46915894021153,14.42803736078717,14.39439258773687,14.37196259643117,14.35327100831251,14.29719624395651,14.24859814336908,14.16261686654431,14.14018701784412,14.03551401029519,13.87102812041423,13.34392539251005,13.11962633508606,13.08224315884873,13.04859824319293,13.01495332753713,12.98504667246286,12.95887856318113,12.94018697506246,12.94018697506246,12.89906553824360,12.85420555563220,12.88785047128800,12.85794410142473,12.86915888316933,12.80560759765027,12.76822442141294,12.70467285068288,12.70467285068288,12.64112156516381,12.62242997704515,12.61121491008954,12.60747664950801,12.60000012834495,12.58504680080781,12.58130854022628]

In [18]:
phis_from_mean = []
errors_phis = []

for i, j, k, l, m, n in zip(x_mean,y_mean,z_mean,x_std,y_std,z_std):
    phis = get_phi((i+j)/2,k,(l+m),n)
    phis_from_mean.append(phis[0])
    errors_phis.append(phis[1])

In [21]:


delta = [i - phis_from_mean[0] for i in phis_from_mean]

steps = np.linspace(200,1200,len(phis_from_mean))

roll_delta = pd.Series(delta).rolling(500, min_periods=0, win_type="hamming").mean()

plt.plot(steps,roll_delta,color='black',label='MLMD')
plt.plot(steps,delta, alpha=0.2,color='black')
plt.xlabel('Temperature (K)')
plt.ylabel(r'$\Delta$ Tilt angle phi ($\degree$)')

plt.ylim(-3, 0.5)
plt.xlim(300, 1000)
plt.yticks([0, -1, -2, -3])
plt.xticks([400, 600, 800, 1000])
plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.1f'))


plt.legend(frameon = False)
plt.savefig("tilting_angle_phi_relative_MLMD_v4.png")
plt.show()

In [13]:
delta_xray_y = [i - xray_y[0] for i in xray_y]
plt.plot(xray_x, delta_xray_y,color='red',marker='o',label='Experiment')

plt.xlabel('Temperature (K)')
plt.ylabel(r'$\Delta$ Tilt angle phi ($\degree$)')


plt.ylim(-3, 0.5)
plt.xlim(300, 1000)
plt.yticks([0, -1, -2, -3])
plt.xticks([400, 600, 800, 1000])
plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.1f'))


plt.legend(frameon = False)
plt.savefig("tilting_angle_phi_relative_Experiment_v3.png")
plt.show()